# Lending Club EDA -- F9 -- Transformation, Feature Diagnostics & Analytical Preparation

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

Diagnostics that inform (but don't perform) cleaning: which log/Box-Cox transforms actually normalize which fields, VIF-based multicollinearity check on the full retained set, and WOE-encoding diagnostics for the categorical features. The actual transformation/encoding is applied in notebooks/03_data_cleaning/, not here -- this notebook only diagnoses what SHOULD be done and why.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

This notebook diagnoses transformation and preparation choices already made
elsewhere (log transforms used in the cleaning notebook, WOE binning used in
notebook 04) -- it checks whether they hold up, it does not apply or change
any transform to persisted data.

| # | What it does |
|---|---|
| 1 | Connect; pull the numeric feature matrix used throughout this EDA suite |
| 2 | Multicollinearity check -- Variance Inflation Factor across numeric features |
| 3 | Does the log transform actually help? Correlation-with-target before vs. after, for every log-transformed field |
| 4 | WOE monotonicity check -- were the bins from notebook 04 actually monotonic, or jagged? |
| 5 | Synthesis -- which transformation choices are validated, which need a second look in the cleaning rebuild |

## Cell 1 -- pull the numeric feature matrix

**What / why:** the diagnostics in this notebook (VIF, correlation-with-
target) need the same numeric feature set already used for IV ranking and
the baseline model, so results here are directly comparable to what's
already been established rather than introducing a new, uncomparable feature
list.

**How:** pull the same 20-column numeric set from `windowed`, median-impute
in memory (not persisted -- same approach as notebook 05).

**Expect:** a complete numeric matrix, no missing values.

In [1]:
import os, duckdb, pandas as pd, numpy as np
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)
con = duckdb.connect("../../data/02_interim/lendingclub.duckdb", read_only=True)

NUMERIC_COLS = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low",
                 "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
                 "revol_bal", "revol_util", "total_acc", "mort_acc",
                 "pub_rec_bankruptcies", "tot_cur_bal", "avg_cur_bal",
                 "bc_open_to_buy", "acc_open_past_24mths",
                 "mo_sin_old_rev_tl_op", "num_actv_rev_tl"]
cols_sql = ", ".join(f"TRY_CAST({c} AS DOUBLE) AS {c}" for c in NUMERIC_COLS)
df = con.sql(f"SELECT {cols_sql}, is_bad FROM windowed").df()
df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(df[NUMERIC_COLS].median())
print(f"matrix ready: {df.shape[0]:,} rows x {len(NUMERIC_COLS)} numeric features")


matrix ready: 1,195,879 rows x 20 numeric features


**What the output shows:** 1,195,879 complete
rows across 20 features, ready for the diagnostics
below.

**Next:** checking multicollinearity first -- several of these features are
conceptually related (`tot_cur_bal`/`avg_cur_bal`, `open_acc`/`total_acc`),
so redundancy is worth quantifying before trusting any individual
coefficient in a linear model.

## Cell 2 -- multicollinearity: Variance Inflation Factor

**What / why:** the baseline logistic regression in notebook 04 reported
individual coefficients, but coefficients from correlated features can be
unstable or misleading -- if two features move together, the model can't
cleanly attribute effect to either one. VIF quantifies this directly: a VIF
above ~5-10 for a feature signals its information is substantially
duplicated elsewhere in the feature set.

**How:** compute VIF for every numeric feature via
`statsmodels.stats.outliers_influence.variance_inflation_factor`.

**Expect:** most features with modest VIF; the current-balance and
account-count fields are the most likely candidates for elevated VIF, since
they're the most conceptually overlapping pairs in the set.

In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

X_vif = StandardScaler().fit_transform(df[NUMERIC_COLS])
vif_data = pd.DataFrame({
    "feature": NUMERIC_COLS,
    "VIF": [variance_inflation_factor(X_vif, i) for i in range(len(NUMERIC_COLS))]
}).sort_values("VIF", ascending=False)
print(vif_data.to_string(index=False))
vif_data.to_csv(os.path.join(ASSETS_TABLES, "eda09_vif_data.csv"), index=False)


             feature      VIF
         tot_cur_bal 6.424792
         avg_cur_bal 5.221603
            open_acc 4.180484
           total_acc 2.650842
     num_actv_rev_tl 2.223379
      fico_range_low 1.993918
          revol_util 1.955423
      bc_open_to_buy 1.824768
pub_rec_bankruptcies 1.791231
             pub_rec 1.724819
acc_open_past_24mths 1.705651
            mort_acc 1.659604
           revol_bal 1.587967
            int_rate 1.441747
           loan_amnt 1.388643
          annual_inc 1.324288
mo_sin_old_rev_tl_op 1.275461
                 dti 1.176796
      inq_last_6mths 1.146976
         delinq_2yrs 1.125759


**What the output shows:**
```
feature      VIF
         tot_cur_bal 6.424792
         avg_cur_bal 5.221603
            open_acc 4.180484
           total_acc 2.650842
     num_actv_rev_tl 2.223379
      fico_range_low 1.993918
          revol_util 1.955423
      bc_open_to_buy 1.824768
pub_rec_bankruptcies 1.791231
             pub_rec 1.724819
acc_open_past_24mths 1.705651
            mort_acc 1.659604
           revol_bal 1.587967
            int_rate 1.441747
           loan_amnt 1.388643
          annual_inc 1.324288
mo_sin_old_rev_tl_op 1.275461
                 dti 1.176796
      inq_last_6mths 1.146976
         delinq_2yrs 1.125759
```
Some features show meaningfully elevated VIF, particularly among the balance/account-count fields -- worth collapsing or dropping one of each highly-correlated pair before Phase 1 modeling, rather than feeding both into a linear model as if they were independent.
Highest VIF: `tot_cur_bal` at 6.4.

**Next:** checking whether the log transforms applied to 16 skewed fields in
the cleaning notebook actually improved their relationship with the target,
rather than assuming skew-reduction automatically means a better feature.

## Cell 3 -- does the log transform actually help?

**What / why:** the cleaning notebook applied a log transform to 16 skewed
fields based on skewness reduction alone (see its cell 4: skew before/after).
Skewness reduction is necessary but not sufficient -- what actually matters
for modeling is whether the transformed feature has a *stronger* linear
relationship with the target than the raw feature. This cell checks that
directly rather than assuming it.

**How:** for every log-transformed field, compute point-biserial correlation
with `is_bad` for both the raw and `log1p`-transformed version, compare.

**Expect:** most fields should show improved (larger-magnitude) correlation
after log transform, since that's usually why skew-reduction helps linear
relationships -- but this is a check, not an assumption, and any field where
the log version does *not* help is worth flagging for the cleaning rebuild.

In [3]:
from scipy import stats as sps

LOG_CANDIDATES = ["annual_inc", "fico_range_low", "delinq_2yrs", "inq_last_6mths",
                   "open_acc", "pub_rec", "revol_bal", "total_acc", "mort_acc",
                   "pub_rec_bankruptcies", "tot_cur_bal", "avg_cur_bal",
                   "bc_open_to_buy", "acc_open_past_24mths",
                   "mo_sin_old_rev_tl_op", "num_actv_rev_tl"]
rows = []
for col in LOG_CANDIDATES:
    raw_corr, _ = sps.pointbiserialr(df["is_bad"], df[col])
    log_corr, _ = sps.pointbiserialr(df["is_bad"], np.log1p(df[col].clip(lower=0)))
    rows.append({"feature": col, "raw_corr": round(raw_corr,4), "log_corr": round(log_corr,4),
                 "improved": abs(log_corr) > abs(raw_corr)})
transform_check = pd.DataFrame(rows).sort_values("feature")
print(transform_check.to_string(index=False))
print(f"\nlog transform improved |correlation| for {transform_check['improved'].sum()} of {len(transform_check)} fields")
transform_check.to_csv(os.path.join(ASSETS_TABLES, "eda09_transform_check.csv"), index=False)


             feature  raw_corr  log_corr  improved
acc_open_past_24mths    0.1054    0.1019     False
          annual_inc   -0.0426   -0.0654      True
         avg_cur_bal   -0.0791   -0.0749     False
      bc_open_to_buy   -0.0811   -0.0781     False
         delinq_2yrs    0.0180    0.0189      True
      fico_range_low   -0.1285   -0.1291      True
      inq_last_6mths    0.0670    0.0680      True
mo_sin_old_rev_tl_op   -0.0524   -0.0637      True
            mort_acc   -0.0766   -0.0821      True
     num_actv_rev_tl    0.0714    0.0692     False
            open_acc    0.0294    0.0291     False
             pub_rec    0.0237    0.0281      True
pub_rec_bankruptcies    0.0238    0.0242      True
           revol_bal   -0.0226   -0.0095     False
         tot_cur_bal   -0.0707   -0.0569     False
           total_acc   -0.0103   -0.0133      True

log transform improved |correlation| for 9 of 16 fields


**What the output shows:**
```
feature  raw_corr  log_corr  improved
acc_open_past_24mths    0.1054    0.1019     False
          annual_inc   -0.0426   -0.0654      True
         avg_cur_bal   -0.0791   -0.0749     False
      bc_open_to_buy   -0.0811   -0.0781     False
         delinq_2yrs    0.0180    0.0189      True
      fico_range_low   -0.1285   -0.1291      True
      inq_last_6mths    0.0670    0.0680      True
mo_sin_old_rev_tl_op   -0.0524   -0.0637      True
            mort_acc   -0.0766   -0.0821      True
     num_actv_rev_tl    0.0714    0.0692     False
            open_acc    0.0294    0.0291     False
             pub_rec    0.0237    0.0281      True
pub_rec_bankruptcies    0.0238    0.0242      True
           revol_bal   -0.0226   -0.0095     False
         tot_cur_bal   -0.0707   -0.0569     False
           total_acc   -0.0103   -0.0133      True

log transform improved |correlation| for 9 of 16 fields
```
The log transform improved the target-correlation magnitude for
9 of 16 fields
-- meaning the blanket log-everything approach should be reconsidered for the fields where it didn't help.
Fields where log did NOT help: acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal -- worth using the raw version (or a different transform) for these specifically in the cleaning rebuild.

**Next:** checking the other transformation choice made elsewhere -- whether
the WOE bins used for the top IV features in notebook 04 were actually
smooth and monotonic, which is what makes a WOE-encoded feature well-behaved
in a linear model.

## Cell 4 -- WOE monotonicity check

**What / why:** notebook 04 computed WOE (weight of evidence) curves for the
top features by IV, using `NTILE(10)` quantile binning. A WOE curve that
moves *monotonically* across bins (risk consistently rises or falls as the
feature increases) is well-behaved for a scorecard-style model; a jagged,
non-monotonic curve usually signals either genuine non-linear risk behavior
or noisy bins that would benefit from coarser binning.

**How:** recompute WOE by decile for the top 3 IV features from notebook 04
(`int_rate`, `grade`-adjacent `sub_grade` if present, `dti`), check whether
WOE strictly increases or decreases bin-over-bin.

**Expect:** `int_rate` should be close to monotonic (it's Lending Club's own
risk score, directly built to correlate with default); noisier features may
show a bin or two out of order.

In [4]:
def woe_by_decile(col):
    q = con.sql(f"""
        WITH binned AS (
            SELECT NTILE(10) OVER (ORDER BY TRY_CAST({col} AS DOUBLE)) AS decile, is_bad
            FROM windowed WHERE TRY_CAST({col} AS DOUBLE) IS NOT NULL
        )
        SELECT decile, count(*) n, sum(is_bad) n_bad, count(*)-sum(is_bad) n_good
        FROM binned GROUP BY 1 ORDER BY 1
    """).df()
    total_bad, total_good = q["n_bad"].sum(), q["n_good"].sum()
    q["good_pct"] = q["n_good"] / total_good
    q["bad_pct"] = q["n_bad"] / total_bad
    q["woe"] = np.log(q["good_pct"] / q["bad_pct"])
    return q

woe_results = {}
for col in ["int_rate", "dti"]:
    w = woe_by_decile(col)
    diffs = w["woe"].diff().dropna()
    monotonic = (diffs >= 0).all() or (diffs <= 0).all()
    woe_results[col] = (w, monotonic)
    print(f"{col}: monotonic across deciles = {monotonic}")
    print(w[["decile","woe"]].to_string(index=False))
    print()


int_rate: monotonic across deciles = True
 decile       woe
      1  1.603772
      2  0.995561
      3  0.613432
      4  0.322053
      5  0.196166
      6 -0.014594
      7 -0.205868
      8 -0.380770
      9 -0.629031
     10 -0.995733

dti: monotonic across deciles = True
 decile       woe
      1  0.390360
      2  0.334083
      3  0.252669
      4  0.174073
      5  0.090843
      6  0.013963
      7 -0.074465
      8 -0.174274
      9 -0.295345
     10 -0.497227



**What the output shows:**
```
int_rate: monotonic across deciles = True
 decile       woe
      1  1.603772
      2  0.995561
      3  0.613432
      4  0.322053
      5  0.196166
      6 -0.014594
      7 -0.205868
      8 -0.380770
      9 -0.629031
     10 -0.995733

dti: monotonic across deciles = True
 decile       woe
      1  0.390360
      2  0.334083
      3  0.252669
      4  0.174073
      5  0.090843
      6  0.013963
      7 -0.074465
      8 -0.174274
      9 -0.295345
     10 -0.497227
```
int_rate and dti came out perfectly monotonic across all 10 deciles.
Both features behave as expected for well-established risk drivers.

**Next:** pulling this notebook's transformation and diagnostic findings
together -- which choices are validated as-is, and which deserve a second
look when the cleaning notebook gets rebuilt with the benefit of all 14 EDA
notebooks' findings.

## Cell 5 -- synthesis

**What / why:** this notebook's whole purpose is to feed directly into the
cleaning-notebook rebuild -- summarizing exactly what to keep, reconsider, or
change is what makes that rebuild grounded in evidence rather than a repeat
of the original guesses.

**How:** a short printed recap referencing the actual diagnostic results
computed above.

**Expect:** a compact action list for the cleaning rebuild.

In [5]:
print("Transformation & feature diagnostics -- summary")
print("="*55)
print(f"Multicollinearity: max VIF = {vif_data['VIF'].max():.1f} ({vif_data.iloc[0]['feature']})")
print(f"Log transforms: improved target-correlation for {transform_check['improved'].sum()}/{len(transform_check)} fields")
if (~transform_check['improved']).any():
    print(f"  -> reconsider log for: {', '.join(transform_check[~transform_check['improved']]['feature'].tolist())}")
print(f"WOE monotonicity: int_rate={woe_results['int_rate'][1]}, dti={woe_results['dti'][1]}")


Transformation & feature diagnostics -- summary
Multicollinearity: max VIF = 6.4 (tot_cur_bal)
Log transforms: improved target-correlation for 9/16 fields
  -> reconsider log for: acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal
WOE monotonicity: int_rate=True, dti=True


**What the output shows:**
```
Transformation & feature diagnostics -- summary
=======================================================
Multicollinearity: max VIF = 6.4 (tot_cur_bal)
Log transforms: improved target-correlation for 9/16 fields
  -> reconsider log for: acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal
WOE monotonicity: int_rate=True, dti=True
```
This is a direct input to the cleaning-notebook rebuild (task 11): the
multicollinearity result flags which balance/account-count features might be
worth collapsing, the log-transform check confirms which fields the existing
approach handles correctly, and the WOE result confirms whether the top risk
drivers behave as expected for scorecard-style modeling.

**Next:** notebook 10 -- checking whether the population itself is stable
across the years and geography represented in this dataset, before treating
any single-year or single-state finding as generally true.